## Seminar 7: Item2item lists

In [ ]:
!pip install implicit

In [133]:
import os
import time
from typing import List, Tuple

import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
import polars as pl
import numpy as np
import scipy.sparse as sp
import seaborn as sns
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight
from implicit.evaluation import mean_average_precision_at_k
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import faiss
import pickle

### 1. Download dataset

The data used in this seminar can be downloaded from

https://huggingface.co/datasets/deepvk/VK-LSVD

We will use VK-LSVD Dataset, which short-video recommendations (like Tik-Tok).

In [134]:
subsample_name = 'up0.001_ip0.001'
content_embedding_size = 32

train_interactions_files = [f'subsamples/{subsample_name}/train/week_{i:02}.parquet'
                            for i in range(25)]
val_interactions_file = [f'subsamples/{subsample_name}/validation/week_25.parquet']

metadata_files = ['metadata/users_metadata.parquet',
                  'metadata/items_metadata.parquet',
                  'metadata/item_embeddings.npz']

for file in (train_interactions_files +
             val_interactions_file +
             metadata_files):
    hf_hub_download(
        repo_id='deepvk/VK-LSVD', repo_type='dataset',
        filename=file, local_dir='VK-LSVD'
    )

train_interactions = pl.concat([pl.scan_parquet(f'VK-LSVD/{file}')
                                for file in train_interactions_files])
train_interactions = train_interactions.collect(engine='streaming')

val_interactions = pl.read_parquet(f'VK-LSVD/{val_interactions_file[0]}')

train_users = train_interactions.select('user_id').unique()
train_items = train_interactions.select('item_id').unique()

item_ids = np.load('VK-LSVD/metadata/item_embeddings.npz')['item_id']
item_embeddings = np.load('VK-LSVD/metadata/item_embeddings.npz')['embedding']

mask = np.isin(item_ids, train_items.to_numpy())
item_ids = item_ids[mask]
item_embeddings = item_embeddings[mask]
item_embeddings = item_embeddings[:, :content_embedding_size]

users_metadata = pl.read_parquet('VK-LSVD/metadata/users_metadata.parquet')
items_metadata = pl.read_parquet('VK-LSVD/metadata/items_metadata.parquet')

users_metadata = users_metadata.join(train_users, on='user_id')
items_metadata = items_metadata.join(train_items, on='item_id')
items_metadata = items_metadata.join(pl.DataFrame({'item_id': item_ids,
                                                   'embedding': item_embeddings}),
                                     on='item_id')


In [135]:
len(train_interactions)

47068641

In [136]:
train_interactions.head()

user_id,item_id,place,platform,agent,timespent,like,dislike,share,bookmark,click_on_author,open_comments
u32,u32,u8,u8,u8,u8,bool,bool,bool,bool,bool,bool
141827770,160593626,1,1,1,56,false,false,false,false,false,false
468779351,327484273,1,0,0,1,false,false,false,false,false,false
494341617,402842289,1,0,0,11,false,false,false,false,false,false
453833313,102672169,0,0,0,45,false,false,false,false,false,false
154047442,139795075,1,0,0,39,false,false,false,false,false,false


In [137]:
items_metadata.head()

item_id,author_id,duration,train_interactions_rank,embedding
u32,u32,u8,u32,"array[f32, 32]"
66761,451170,92,11105,"[-0.53418, -0.093689, … 0.083801]"
71482,421359,14,19098,"[-0.548828, 0.061951, … 0.110901]"
99494,396985,7,10228,"[-0.571777, 0.30249, … 0.128784]"
150686,674084,33,6408,"[-0.176025, -0.343506, … 0.093628]"
182273,882856,32,14982,"[-0.21875, 0.160278, … 0.203369]"


### 2. Features calculation

In [ ]:
### Calculation of counters (may be helpful in your further researches) ###

# counters_df = train_interactions.group_by("item_id").agg([
#     pl.col("like").sum().alias("likes"),
#     pl.col("dislike").sum().alias("dislikes"),
#     pl.col("share").sum().alias("shares"),
#     pl.col("open_comments").sum().alias("open_comments"),
#     pl.col("item_id").count().alias("shows"),

#     # Ratios
#     (pl.col("like").sum() / pl.col("item_id").count()).alias("like_rate"),
#     (pl.col("share").sum() / pl.col("item_id").count()).alias("share_rate"),
#     (pl.col("open_comments").sum() / pl.col("item_id").count()).alias("open_comments_rate"),
#     (pl.col("like").sum() / (pl.col("dislike").sum() + 1)).alias("like_dislike_ratio")
# ])

In [ ]:
# counters_df.head()

Let's create joined dataset with events and items metadata which will be helpful for further application:

In [138]:
# Info from user interactions
train_pairs = train_interactions.select(
    [
        "user_id",
        "item_id",
        "like",
        "dislike",
        "share",
        "open_comments", 
        "timespent"
    ]
)
cast_schema = {
    "like": pl.Float32,
    "dislike": pl.Float32, 
    "share": pl.Float32,
    "open_comments": pl.Float32
}
train_pairs = train_pairs.with_columns([
    pl.col(col).cast(dtype) for col, dtype in cast_schema.items()
])

# Info from items metadata
item_info = items_metadata.select(["item_id", "embedding", "duration"])

joined = train_pairs.join(item_info, on="item_id")

In [139]:
joined.head()

user_id,item_id,like,dislike,share,open_comments,timespent,embedding,duration
u32,u32,f32,f32,f32,f32,u8,"array[f32, 32]",u8
141827770,160593626,0.0,0.0,0.0,0.0,56,"[-0.524902, 0.065979, … -0.15271]",83
468779351,327484273,0.0,0.0,0.0,0.0,1,"[-0.390869, 0.140869, … -0.109192]",12
494341617,402842289,0.0,0.0,0.0,0.0,11,"[-0.47876, 0.200317, … -0.086121]",11
453833313,102672169,0.0,0.0,0.0,0.0,45,"[-0.28125, -0.148804, … -0.026016]",45
154047442,139795075,0.0,0.0,0.0,0.0,39,"[-0.658203, 0.168579, … -0.009827]",14


Let's create composite target which combine explicit and implicit feedbacks for items:

In [ ]:
def create_composite_target(df, weights=None):
    if weights is None:
        weights = {
            'like': 3.0,
            'dislike': -2.0,
            'share': 4.0,
            'open_comments': 2.0,
            'completion_rate': 1.5,
        }
    
    return (
        df['like'] * weights['like'] +
        df['dislike'] * weights['dislike'] +
        df['share'] * weights['share'] +
        df['open_comments'] * weights['open_comments'] +
        df['timespent'] / df['duration'] * weights['completion_rate']
    )

joined = joined.with_columns([
    create_composite_target(joined).alias("fat"), # Feedback and Time
    (joined['timespent'] / joined['duration']).alias("norm_ts")
])

In [141]:
joined.head()

user_id,item_id,like,dislike,share,open_comments,timespent,embedding,duration,fat,norm_ts
u32,u32,f32,f32,f32,f32,u8,"array[f32, 32]",u8,f64,f64
141827770,160593626,0.0,0.0,0.0,0.0,56,"[-0.524902, 0.065979, … -0.15271]",83,1.012048,0.674699
468779351,327484273,0.0,0.0,0.0,0.0,1,"[-0.390869, 0.140869, … -0.109192]",12,0.125,0.083333
494341617,402842289,0.0,0.0,0.0,0.0,11,"[-0.47876, 0.200317, … -0.086121]",11,1.5,1.0
453833313,102672169,0.0,0.0,0.0,0.0,45,"[-0.28125, -0.148804, … -0.026016]",45,1.5,1.0
154047442,139795075,0.0,0.0,0.0,0.0,39,"[-0.658203, 0.168579, … -0.009827]",14,4.178571,2.785714


### 3. ALS: train and use

Let's train several ALS models for further usage as similarity features.

Firstly, we need to construct mappings and prepare interaction matrices:

In [142]:
def create_mappings(df):
    unique_users = df["user_id"].unique().sort()
    unique_items = df["item_id"].unique().sort()
    
    user_to_idx = {user_id: idx for idx, user_id in enumerate(unique_users)}
    item_to_idx = {item_id: idx for idx, item_id in enumerate(unique_items)}
    idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}
    idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}
    
    return user_to_idx, item_to_idx, idx_to_user, idx_to_item


user_to_idx, item_to_idx, idx_to_user, idx_to_item = create_mappings(joined)

print(f"Number of users: {len(user_to_idx)}")
print(f"Number of items: {len(item_to_idx)}")

Number of users: 10000
Number of items: 19628


In [143]:
def prepare_interaction_matrix(df, user_to_idx, item_to_idx, target):
    users = [user_to_idx[user_id] for user_id in df["user_id"].to_list()]
    items = [item_to_idx[item_id] for item_id in df["item_id"].to_list()]
    interactions = df[target].to_list()
    
    interaction_matrix = csr_matrix(
        (interactions, (users, items)),
        shape=(len(user_to_idx), len(item_to_idx))
    )
    
    return interaction_matrix


norm_ts_matrix = prepare_interaction_matrix(
    joined,
    user_to_idx,
    item_to_idx,
    "norm_ts"
)

ts_matrix = prepare_interaction_matrix(
    joined,
    user_to_idx,
    item_to_idx,
    "timespent"
)

fat_matrix = prepare_interaction_matrix(
    joined,
    user_to_idx,
    item_to_idx,
    "fat"
)


print(f"Interaction matrix shape: {norm_ts_matrix.shape}")

Interaction matrix shape: (10000, 19628)


In [144]:
def train_als_model(interaction_matrix, factors=32, iterations=15, regularization=0.01):
    """
    Train ALS model with BM25 weighting
    """
    weighted_matrix = bm25_weight(interaction_matrix, K1=100, B=0.8)
    
    model = AlternatingLeastSquares(
        factors=factors,
        iterations=iterations,
        regularization=regularization,
        random_state=42
    )
    
    model.fit(weighted_matrix)
    
    return model


norm_ts_als = train_als_model(norm_ts_matrix, factors=32)
ts_als = train_als_model(ts_matrix, factors=32)
fat_als = train_als_model(fat_matrix, factors=32)

/Users/vl.naumov/anaconda3/envs/recsys/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.26685428619384766 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

/Users/vl.naumov/anaconda3/envs/recsys/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.2619822025299072 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

/Users/vl.naumov/anaconda3/envs/recsys/lib/python3.10/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.2852051258087158 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

In [145]:
def create_embedding_mapping(model, idx_to_item):
    return {item_id: model.item_factors[idx] for idx, item_id in idx_to_item.items() 
            if idx < len(model.item_factors)}

# item embeddings mapping
norm_ts_als_item_emb_mapping = create_embedding_mapping(norm_ts_als, idx_to_item)
ts_als_item_emb_mapping = create_embedding_mapping(ts_als, idx_to_item)
fat_als_item_emb_mapping = create_embedding_mapping(fat_als, idx_to_item)


# with open('norm_ts_als_item_emb_mapping.pkl', 'rb') as file:
#     norm_ts_als_item_emb_mapping = pickle.load(file)

# with open('ts_als_item_emb_mapping.pkl', 'rb') as file:
#     ts_als_item_emb_mapping = pickle.load(file)

# with open('fat_als_item_emb_mapping.pkl', 'rb') as file:
#     fat_als_item_emb_mapping = pickle.load(file)

In [ ]:
# import pickle 

# pickle.dump(fat_als, open("fat_als.pkl", "wb"))

In [146]:
als_dfs = {}
for source, mapping in [
    ("norm_ts", norm_ts_als_item_emb_mapping),
    ("ts", ts_als_item_emb_mapping), 
    ("fat", fat_als_item_emb_mapping)
]:
    als_dfs[source] = pl.DataFrame({
        "item_id": list(mapping.keys()),
        f"embedding_{source}": [
            (embedding / np.linalg.norm(embedding)).astype(np.float32).tolist() 
            for embedding in mapping.values()
        ]
    }).cast({
        "item_id": pl.Int64,
        f"embedding_{source}": pl.Array(pl.Float32, 32)
    }).sort(by="item_id")


all_als_embeddings = als_dfs["norm_ts"]
for source in ["ts", "fat"]:
    all_als_embeddings = all_als_embeddings.join(
        als_dfs[source], on="item_id", how="full", suffix=f"_{source}"
    ).drop([f"item_id_{source}"])

In [147]:
all_als_embeddings.head()

item_id,embedding_norm_ts,embedding_ts,embedding_fat
i64,"array[f32, 32]","array[f32, 32]","array[f32, 32]"
66761,"[-0.227943, 0.111285, … -0.186449]","[-0.17403, -0.178733, … -0.12163]","[-0.283137, 0.156175, … -0.196496]"
71482,"[0.041034, -0.044472, … -0.022994]","[-0.349921, -0.137602, … -0.022903]","[0.034006, -0.045101, … -0.023397]"
99494,"[0.117914, -0.049371, … 0.116436]","[0.205214, -0.101462, … -0.030804]","[0.109607, -0.032485, … 0.114828]"
150686,"[-0.05078, -0.033754, … 0.193209]","[0.150414, -0.304521, … 0.466781]","[-0.063437, -0.046163, … 0.22253]"
182273,"[-0.094189, -0.06345, … -0.105585]","[-0.105264, 0.090803, … 0.077165]","[-0.146296, -0.015182, … -0.102134]"


In [148]:
all_als_embeddings = pl.read_parquet("als_item_embeddings.parquet")

### 4. Constructing kNN indices

In [149]:
items_metadata.sort("item_id")

item_id,author_id,duration,train_interactions_rank,embedding
u32,u32,u8,u32,"array[f32, 32]"
66761,451170,92,11105,"[-0.53418, -0.093689, … 0.083801]"
71482,421359,14,19098,"[-0.548828, 0.061951, … 0.110901]"
99494,396985,7,10228,"[-0.571777, 0.30249, … 0.128784]"
150686,674084,33,6408,"[-0.176025, -0.343506, … 0.093628]"
182273,882856,32,14982,"[-0.21875, 0.160278, … 0.203369]"
…,…,…,…,…
607900487,1215678,42,4335,"[-0.514648, 0.009583, … 0.111145]"
607918712,1209115,165,9277,"[-0.097656, 0.04248, … -0.013161]"
607945053,994030,89,12921,"[-0.458252, -0.196045, … 0.132568]"


In [150]:
content_emb_mapping = {row[0]: np.array(row[1]) for row in items_metadata.select(["item_id", "embedding"]).iter_rows()}

In [151]:
content_emb = items_metadata["embedding"].to_numpy()
als_norm_ts_emb = all_als_embeddings["embedding_norm_ts"].to_numpy()
als_ts_emb = all_als_embeddings["embedding_ts"].to_numpy()
als_fat_emb = all_als_embeddings["embedding_fat"].to_numpy()

In [152]:
content_emb.shape

(19628, 32)

In [153]:
def create_index(emb):
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index

In [154]:
content_index = create_index(content_emb)
als_norm_ts_index = create_index(als_norm_ts_emb)
als_ts_index = create_index(als_ts_emb)
als_fat_index = create_index(als_fat_emb)

In [155]:
nn = 11 # except itself!

content_D, content_I = content_index.search(content_emb, nn)
als_norm_ts_D, als_norm_ts_I = als_norm_ts_index.search(als_norm_ts_emb, nn)
als_ts_index_D, als_ts_index_I = als_ts_index.search(als_ts_emb, nn)
als_fat_index_D, als_fat_index_I = als_fat_index.search(als_fat_emb, nn)

### 5. Relevance model

In [179]:
train_df = pl.read_parquet("relevance_train.parquet")
test_df = pl.read_parquet("relevance_test.parquet")

print(f"Train: {len(train_df)} pairs")
print(f"Test: {len(test_df)} pairs")
print("Label distribution:")
print(train_df["label"].value_counts())

Train: 64000 pairs
Test: 10000 pairs
Label distribution:
shape: (2, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 1     ┆ 32000 │
│ 0     ┆ 32000 │
└───────┴───────┘


In [180]:
train_df.head()

left_item_id,right_item_id,label
i64,i64,i64
494515212,449731463,1
189684680,340148308,1
599031915,548889785,1
424604732,359230130,1
481800635,360753001,1


In [181]:
def calculate_cosine_similarity(emb1, emb2):
    """Calculate cosine similarity between two embeddings"""
    if emb1 is None or emb2 is None:
        return 0.0
    return float(np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2)))

def add_rel_cosine_features(pairs_df, emb_mappings):
    """
    Add cosine similarity features for all embedding types
    """
    features_df = pairs_df.with_columns([
        # Cosine similarity for content embeddings
        pl.struct(["left_item_id", "right_item_id"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["content"].get(x["left_item_id"]),
                emb_mappings["content"].get(x["right_item_id"])
            ), return_dtype=pl.Float64
        ).alias("cos_content"),
        
        # Cosine similarity for norm_ts embeddings
        pl.struct(["left_item_id", "right_item_id"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["norm_ts"].get(x["left_item_id"]),
                emb_mappings["norm_ts"].get(x["right_item_id"])
            ), return_dtype=pl.Float64
        ).alias("cos_norm_ts"),
        
        # Cosine similarity for ts embeddings
        pl.struct(["left_item_id", "right_item_id"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["ts"].get(x["left_item_id"]),
                emb_mappings["ts"].get(x["right_item_id"])
            ), return_dtype=pl.Float64
        ).alias("cos_ts"),

        # Cosine similarity for ts embeddings
        pl.struct(["left_item_id", "right_item_id"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["fat"].get(x["left_item_id"]),
                emb_mappings["fat"].get(x["right_item_id"])
            ), return_dtype=pl.Float64
        ).alias("cos_fat"),
    ])
    
    return features_df


emb_mappings = {
    "content": content_emb_mapping,
    "norm_ts": norm_ts_als_item_emb_mapping,
    "ts": ts_als_item_emb_mapping,
    "fat": fat_als_item_emb_mapping
}

train_features = add_rel_cosine_features(train_df, emb_mappings)
test_features = add_rel_cosine_features(test_df, emb_mappings)

print("Train features:")
print(train_features.head())
print("\nTest features:")
print(test_features.head())

Train features:
shape: (5, 7)
┌──────────────┬───────────────┬───────┬─────────────┬─────────────┬──────────┬──────────┐
│ left_item_id ┆ right_item_id ┆ label ┆ cos_content ┆ cos_norm_ts ┆ cos_ts   ┆ cos_fat  │
│ ---          ┆ ---           ┆ ---   ┆ ---         ┆ ---         ┆ ---      ┆ ---      │
│ i64          ┆ i64           ┆ i64   ┆ f64         ┆ f64         ┆ f64      ┆ f64      │
╞══════════════╪═══════════════╪═══════╪═════════════╪═════════════╪══════════╪══════════╡
│ 494515212    ┆ 449731463     ┆ 1     ┆ 0.904345    ┆ 0.537225    ┆ 0.530534 ┆ 0.582436 │
│ 189684680    ┆ 340148308     ┆ 1     ┆ 0.842051    ┆ 0.231645    ┆ 0.242136 ┆ 0.264549 │
│ 599031915    ┆ 548889785     ┆ 1     ┆ 0.930742    ┆ -0.030737   ┆ -0.12286 ┆ 0.006053 │
│ 424604732    ┆ 359230130     ┆ 1     ┆ 0.864512    ┆ 0.243816    ┆ 0.024466 ┆ 0.268745 │
│ 481800635    ┆ 360753001     ┆ 1     ┆ 0.860658    ┆ 0.218726    ┆ 0.109953 ┆ 0.240861 │
└──────────────┴───────────────┴───────┴─────────────┴──────

Let's use CatBoost for models!

In [160]:
import catboost as cb
from catboost import Pool

In [162]:
def prepare_catboost_data(features_df, features_cols):
    X = features_df.select(features_cols).to_numpy()
    y = features_df["label"].to_numpy()
    
    return X, y


features_cols = ["cos_norm_ts", "cos_ts", "cos_content"]
X_train, y_train = prepare_catboost_data(train_features, features_cols)
X_test, y_test = prepare_catboost_data(test_features, features_cols)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

train_pool = Pool(X_train, y_train, feature_names=features_cols)
test_pool = Pool(X_test, y_test, feature_names=features_cols)

catboost_model = cb.CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100
)

catboost_model.fit(
    train_pool,
    eval_set=test_pool,
    plot=True,
    verbose=True
)

X_train shape: (64000, 3)
X_test shape: (10000, 3)


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.8914608	best: 0.8914608 (0)	total: 15.8ms	remaining: 3.14s
1:	test: 0.8981123	best: 0.8981123 (1)	total: 22.1ms	remaining: 2.18s
2:	test: 0.8994182	best: 0.8994182 (2)	total: 27.5ms	remaining: 1.8s
3:	test: 0.9007883	best: 0.9007883 (3)	total: 31.7ms	remaining: 1.55s
4:	test: 0.9014269	best: 0.9014269 (4)	total: 36.1ms	remaining: 1.41s
5:	test: 0.9015476	best: 0.9015476 (5)	total: 40.6ms	remaining: 1.31s
6:	test: 0.9018120	best: 0.9018120 (6)	total: 45.8ms	remaining: 1.26s
7:	test: 0.9021900	best: 0.9021900 (7)	total: 51.2ms	remaining: 1.23s
8:	test: 0.9022177	best: 0.9022177 (8)	total: 56.1ms	remaining: 1.19s
9:	test: 0.9022963	best: 0.9022963 (9)	total: 60.6ms	remaining: 1.15s
10:	test: 0.9023473	best: 0.9023473 (10)	total: 66.8ms	remaining: 1.15s
11:	test: 0.9022555	best: 0.9023473 (10)	total: 71.6ms	remaining: 1.12s
12:	test: 0.9023035	best: 0.9023473 (10)	total: 77.3ms	remaining: 1.11s
13:	test: 0.9022749	best: 0.9023473 (10)	total: 83.8ms	remaining: 1.11s
14:	test: 0.9

Let's also save model:

In [163]:
catboost_model.save_model("rel_model.cbm")

### 6. Attractivity model

In [164]:
train_df = pl.read_parquet("attractivity_train.parquet")
test_df = pl.read_parquet("attractivity_test.parquet")

print(f"Train: {len(train_df)} pairs")
print(f"Test: {len(test_df)} pairs")

Train: 192000 pairs
Test: 48000 pairs


In [165]:
train_df.head()

user_id,winner_item,loser_item,weight,winner_time,loser_time,winner_log_time,loser_log_time,winner_quartile,loser_quartile
i64,i64,i64,f64,i64,i64,f64,f64,i64,i64
61524223,239018670,460307912,1.185624,179,54,5.192957,4.007333,1,2
61524223,402859747,574786450,1.185624,179,54,5.192957,4.007333,1,2
61524223,420903135,9927215,1.140161,171,54,5.147494,4.007333,1,2
61524223,493820017,413929221,1.116631,167,54,5.123964,4.007333,1,2
61524223,239018670,351272085,1.529395,179,38,5.192957,3.663562,1,3


In [ ]:
def add_pairwise_features(pairs_df, emb_mappings):
    features_df = pairs_df.with_columns([
        pl.struct(["winner_item", "loser_item"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["norm_ts"].get(x["winner_item"]),
                emb_mappings["norm_ts"].get(x["loser_item"])
            ), return_dtype=pl.Float64
        ).alias("cos_norm_ts"),
        
        pl.struct(["winner_item", "loser_item"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["ts"].get(x["winner_item"]),
                emb_mappings["ts"].get(x["loser_item"])
            ), return_dtype=pl.Float64
        ).alias("cos_ts"),
        
        pl.struct(["winner_item", "loser_item"]).map_elements(
            lambda x: calculate_cosine_similarity(
                emb_mappings["fat"].get(x["winner_item"]),
                emb_mappings["fat"].get(x["loser_item"])
            ), return_dtype=pl.Float64
        ).alias("cos_fat"),
        
    ]).fill_nan(0).fill_null(0)

    return features_df


emb_mappings_rank = {
    "norm_ts": norm_ts_als_item_emb_mapping,
    "ts": ts_als_item_emb_mapping,
    "fat": fat_als_item_emb_mapping
}

    
train_features = add_pairwise_features(train_df, emb_mappings_rank)
test_features = add_pairwise_features(test_df, emb_mappings_rank)

In [169]:
feature_cols = [
    'cos_norm_ts', 'cos_ts', 'cos_fat'
]

def create_pairwise_pool(df, feature_cols):
    X = df.select(feature_cols).to_numpy()
    y = df["weight"].to_numpy()
    
    # Group ID: each user forms a group for pairwise comparisons
    group_id = df["user_id"].to_list()
    
    return Pool(
        data=X,
        label=y,
        group_id=group_id,
        feature_names=feature_cols
    )

train_pool = create_pairwise_pool(train_features, feature_cols)
test_pool = create_pairwise_pool(test_features, feature_cols)

print(f"Train pool: {train_pool.num_row()} pairs, {len(set(train_pool.get_group_id_hash()))} user groups")
print(f"Test pool: {test_pool.num_row()} pairs, {len(set(test_pool.get_group_id_hash()))} user groups")

Train pool: 192000 pairs, 8000 user groups
Test pool: 48000 pairs, 2000 user groups


In [170]:
model = cb.CatBoostRanker(
    iterations=100,
    learning_rate=0.1,
    depth=10,
    loss_function='PairLogitPairwise',
    eval_metric='PairAccuracy',
    random_seed=42,
    early_stopping_rounds=50,
    verbose=100,
    task_type='CPU',
    bootstrap_type='Bernoulli',
    subsample=0.8
)

model.fit(
    train_pool,
    eval_set=test_pool,
    plot=True,
    verbose=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.4637952	test: 0.4573690	best: 0.4573690 (0)	total: 9.72s	remaining: 16m 2s
1:	learn: 0.4675366	test: 0.4615803	best: 0.4615803 (1)	total: 18.6s	remaining: 15m 11s
2:	learn: 0.4776430	test: 0.4722674	best: 0.4722674 (2)	total: 27.3s	remaining: 14m 43s
3:	learn: 0.4774166	test: 0.4720071	best: 0.4722674 (2)	total: 35.6s	remaining: 14m 15s
4:	learn: 0.4842064	test: 0.4787984	best: 0.4787984 (4)	total: 43.8s	remaining: 13m 52s
5:	learn: 0.4888236	test: 0.4832721	best: 0.4832721 (5)	total: 51.8s	remaining: 13m 32s
6:	learn: 0.4909630	test: 0.4854320	best: 0.4854320 (6)	total: 59.9s	remaining: 13m 15s
7:	learn: 0.4907574	test: 0.4854300	best: 0.4854320 (6)	total: 1m 8s	remaining: 13m 4s
8:	learn: 0.4933195	test: 0.4880278	best: 0.4880278 (8)	total: 1m 16s	remaining: 12m 53s
9:	learn: 0.4941707	test: 0.4890397	best: 0.4890397 (9)	total: 1m 24s	remaining: 12m 39s
10:	learn: 0.4942220	test: 0.4887596	best: 0.4890397 (9)	total: 1m 32s	remaining: 12m 26s
11:	learn: 0.4958348	test: 0.4

In [171]:
model.save_model("attr_model.cbm")

### 7. Full pipeline

Let's apply both models for content index pairs:

In [172]:
large_content_D, large_content_I = content_index.search(content_emb, 51)

Let's start with **relevance** model:

In [173]:
item_to_candidates = {}

for i in range(len(large_content_I)):
    item_id = idx_to_item[i]
    candidate_indices = large_content_I[i, 1:]
    candidates = [idx_to_item[idx] for idx in candidate_indices]
    item_to_candidates[item_id] = candidates

print(f"Created candidate mapping for {len(item_to_candidates)} items")

Created candidate mapping for 19628 items


In [175]:
pair_data = {
    "left_item_id": [],
    "right_item_id": []
}

for anchor_item, candidates in item_to_candidates.items():
    for candidate in candidates:
        pair_data["left_item_id"].append(anchor_item)
        pair_data["right_item_id"].append(candidate)

pairs_df = pl.DataFrame(pair_data)
print(f"Created {len(pairs_df)} anchor-candidate pairs")

Created 981400 anchor-candidate pairs


In [176]:
pairs_df.head()

left_item_id,right_item_id
i64,i64
66761,109106364
66761,557978018
66761,388326120
66761,449499882
66761,483575900


In [182]:
rel_features = add_rel_cosine_features(pairs_df, emb_mappings)

In [185]:
model = cb.CatBoostClassifier()
model.load_model("rel_model.cbm")


predictions = model.predict(rel_features.to_numpy())

probabilities = model.predict_proba(rel_features.to_numpy())


result_df = rel_features.with_columns([
    pl.Series("prediction", predictions),
    pl.Series("probability", probabilities[:, 1])
])

In [186]:
result_df

left_item_id,right_item_id,cos_content,cos_norm_ts,cos_ts,cos_fat,prediction,probability
i64,i64,f64,f64,f64,f64,i64,f64
66761,109106364,0.870151,0.509082,0.442954,0.494225,1,0.671198
66761,557978018,0.828392,0.229613,0.274079,0.250548,0,0.219806
66761,388326120,0.845043,0.629529,0.577662,0.603832,0,0.360925
66761,449499882,0.806064,0.015063,-0.094548,0.065023,0,0.023249
66761,483575900,0.7805,0.329705,0.341881,0.36069,0,0.002365
…,…,…,…,…,…,…,…
608034240,347891187,0.744128,0.238893,0.251291,0.218722,0,0.000583
608034240,405831926,0.695546,-0.02686,-0.006442,-0.019551,0,0.000587
608034240,476174248,0.762202,0.015366,0.109446,0.040314,0,0.000884


In [188]:
filtered_df = result_df.filter(pl.col("probability") > 0.5)

In [189]:
filtered_df.head()

left_item_id,right_item_id,cos_content,cos_norm_ts,cos_ts,cos_fat,prediction,probability
i64,i64,f64,f64,f64,f64,i64,f64
66761,109106364,0.870151,0.509082,0.442954,0.494225,1,0.671198
99494,430275926,0.932178,0.774266,0.849634,0.794558,1,0.905478
99494,328453816,0.878107,0.375147,0.394275,0.386487,1,0.728872
182273,182273,1.0,1.0,1.0,1.0,1,0.770477
187690,433579385,0.866795,0.518749,0.657678,0.514823,1,0.607496


In [190]:
filtered_mapping = (
    filtered_df
    .group_by("left_item_id")
    .agg(pl.col("right_item_id"))
    .to_pandas()
    .set_index("left_item_id")["right_item_id"]
    .to_dict()
)

In [191]:
filtered_mapping

{148333465: array([449461118, 247466558, 524926855, 280113049, 253202289, 178694425,
        605915601]),
 213913267: array([139541746]),
 556536638: array([556536638]),
 417088444: array([ 11867186, 528157114, 222045211,   1474043, 508112661, 194238572,
        540418499, 465630365, 251858472]),
 335178845: array([345890138, 447033500,  94224488, 404600671, 212280067, 351291260,
         29739794]),
 69905978: array([ 69905978, 130921255, 449747553, 291668056, 439900213, 499720927,
         38108978, 390310976, 393093070, 261493219, 424535058, 530371491,
        360544803]),
 17061251: array([492244740,   1514636, 124311936, 179438942, 170813513, 360784975]),
 262091056: array([127143490, 466317504,  64453122, 553954552, 288807941, 563704658,
        421672589, 458638756,  16390185, 382200604, 233948848,  13310481,
        351322830, 383842885, 386863330, 303651369, 343226832, 465627925,
        112622435,  41962692]),
 317269279: array([562478476, 406917211]),
 528448575: array([4378

In [192]:
print(len(filtered_mapping))

11498


Then continue with **attractivity** model:

In [193]:
ranking_data = []

for left_item_id, candidates in filtered_mapping.items():
    if len(candidates) < 2:
        continue
        
    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):
            ranking_data.append({
                "left_item_id": left_item_id,
                "winner_item": candidates[i],
                "loser_item": candidates[j]
            })

ranking_df = pl.DataFrame(ranking_data)
print(f"Created {len(ranking_df)} ranking pairs")

Created 561223 ranking pairs


In [194]:
ranking_df.head()

left_item_id,winner_item,loser_item
i64,i64,i64
148333465,449461118,247466558
148333465,449461118,524926855
148333465,449461118,280113049
148333465,449461118,253202289
148333465,449461118,178694425


In [195]:
attr_features = add_pairwise_features(ranking_df, emb_mappings_rank)

In [196]:
attr_features.head()

left_item_id,winner_item,loser_item,cos_norm_ts,cos_ts,cos_fat
i64,i64,i64,f64,f64,f64
148333465,449461118,247466558,0.41032,0.567779,0.411441
148333465,449461118,524926855,0.396193,0.573492,0.394636
148333465,449461118,280113049,0.77096,0.728697,0.782349
148333465,449461118,253202289,-0.041717,0.101756,-0.004464
148333465,449461118,178694425,0.790139,0.784434,0.793927


In [197]:
features_cols = ["cos_norm_ts", "cos_ts", "cos_fat"]
ranking_pool = Pool(
    data=attr_features.select(features_cols).to_numpy(),
    group_id=attr_features["left_item_id"].to_list(),
    feature_names=feature_cols
)

ranker = cb.CatBoostRanker()
ranker.load_model("attr_model.cbm")
ranking_scores = ranker.predict(ranking_pool)

ranking_results = attr_features.with_columns([
    pl.Series("ranking_score", ranking_scores)
])

In [198]:
ranking_results.head(100)

left_item_id,winner_item,loser_item,cos_norm_ts,cos_ts,cos_fat,ranking_score
i64,i64,i64,f64,f64,f64,f64
148333465,449461118,247466558,0.41032,0.567779,0.411441,-0.013988
148333465,449461118,524926855,0.396193,0.573492,0.394636,-0.041119
148333465,449461118,280113049,0.77096,0.728697,0.782349,-0.251018
148333465,449461118,253202289,-0.041717,0.101756,-0.004464,0.015466
148333465,449461118,178694425,0.790139,0.784434,0.793927,-0.251018
…,…,…,…,…,…,…
69905978,130921255,390310976,0.800788,0.868714,0.804149,-0.251018
69905978,130921255,393093070,0.854466,0.867384,0.852286,-0.251018
69905978,130921255,261493219,0.782536,0.800515,0.785692,-0.251018


In [199]:
stats = ranking_results.select([
    pl.col("ranking_score").mean().alias("mean_score"),
    pl.col("ranking_score").min().alias("min_score"), 
    pl.col("ranking_score").max().alias("max_score"),
    pl.col("ranking_score").std().alias("std_score")
])

print("Ranking score statistics:")
print(stats)

Ranking score statistics:
shape: (1, 4)
┌────────────┬───────────┬───────────┬───────────┐
│ mean_score ┆ min_score ┆ max_score ┆ std_score │
│ ---        ┆ ---       ┆ ---       ┆ ---       │
│ f64        ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═══════════╪═══════════╡
│ -0.084476  ┆ -0.475545 ┆ 0.598618  ┆ 0.116736  │
└────────────┴───────────┴───────────┴───────────┘


Finally, let's rerank candidates and get final mapping:

In [200]:
reranked_mapping = {}

for left_item_id in ranking_results["left_item_id"].unique():
    group_pairs = ranking_results.filter(pl.col("left_item_id") == left_item_id)
    
    all_candidates = set()
    for row in group_pairs.iter_rows(named=True):
        all_candidates.add(row["winner_item"])
        all_candidates.add(row["loser_item"])
    
    win_counts = {candidate: 0 for candidate in all_candidates}
    
    for row in group_pairs.iter_rows(named=True):
        winner = row["winner_item"]
        loser = row["loser_item"]
        score = row["ranking_score"]
        
        if score > 0:
            win_counts[winner] += 1
        else:
            win_counts[loser] += 1
    
    ranked_candidates = sorted(win_counts.keys(), key=lambda x: win_counts[x], reverse=True)
    reranked_mapping[left_item_id] = ranked_candidates

print(f"Created reranked mapping for {len(reranked_mapping)} items")

Created reranked mapping for 7963 items


In [201]:
filtered_mapping[146139702]

array([531058869, 594016229, 224886878, 487828536, 213134397, 594913826,
        80490689,  36664859, 467880559, 182528742, 554527330, 378785249])

In [202]:
reranked_mapping[146139702]

[554527330,
 182528742,
 378785249,
 467880559,
 594913826,
 487828536,
 36664859,
 80490689,
 213134397,
 531058869,
 224886878,
 594016229]